In [ ]:
# 저장된 v2 답변을 채점기 v3로 재채점합니다.
# 생성 모델과 검색기를 다시 실행하지 않습니다.
# v2 실험의 동일한 79개 답변으로 채점 로직 변경 효과만 비교합니다.
# v3 변경: 기권 표현, Kiwi 조사/어미, 백분율 및 금액 표기 동치 처리.
print('채점기 v3 재채점 노트북을 시작합니다.')

In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.home() / 'sprint-public-procurement-rag-assistant'
V2_RUN_ID = '20260909T075822Z'
EXPECTED_SCORER_VERSION = '3.0.1'
EXPECTED_SCORER_SHA256 = 'd1a6f2ab641b690525d8a4b2d38d8bd3d3aaa838c480d4ef29c353654b271a2c'
V2_RUN_DIR = PROJECT_ROOT / 'output' / 'scoring_v2_runs' / V2_RUN_ID
V3_SCORER_PATH = PROJECT_ROOT / 'src' / 'evaluation' / 'scoring_v3' / 'scorer.py'
V3_BASE_PATH = PROJECT_ROOT / 'src' / 'evaluation' / 'scoring_v3' / '_base.py'

required = [
    V2_RUN_DIR / 'inference.jsonl',
    V2_RUN_DIR / 'scoring_details.jsonl',
    V2_RUN_DIR / 'summary.json',
    V3_SCORER_PATH,
    V3_BASE_PATH,
    PROJECT_ROOT / 'output' / 'merged_docs.pkl',
    PROJECT_ROOT / 'data' / 'golden_set_v3' / 'rag-56.draft.jsonl',
    PROJECT_ROOT / 'data' / 'golden_set_v3' / 'set-13.draft.jsonl',
    PROJECT_ROOT / 'data' / 'golden_set_v3' / 'document-structure-visual-qa.jsonl',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('필수 파일이 없습니다:\n' + '\n'.join(missing))

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('입력 v2 결과:', V2_RUN_DIR)
print('생성 API 재호출: 안 함')

In [ ]:
import hashlib
import importlib.util
import json
import subprocess
from datetime import datetime, timezone

import pandas as pd
from src.evaluation.golden_set_v3 import load_golden_set_v3

spec = importlib.util.spec_from_file_location('scorer_v3', V3_SCORER_PATH)
scorer_v3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer_v3)

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8-sig').splitlines() if line.strip()]

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

loaded_scorer_sha256 = sha256_file(V3_SCORER_PATH)
if scorer_v3.SCORER_VERSION != EXPECTED_SCORER_VERSION:
    raise RuntimeError(f'채점기 버전 불일치: {scorer_v3.SCORER_VERSION} != {EXPECTED_SCORER_VERSION}')
if loaded_scorer_sha256 != EXPECTED_SCORER_SHA256:
    raise RuntimeError(f'채점기 해시 불일치: {loaded_scorer_sha256}')
print('채점기:', scorer_v3.SCORER_VERSION, loaded_scorer_sha256)
print('Kiwi 점검:', scorer_v3.morph_tokens('GKL 그룹웨어 사업이 더 큽니다.'))

In [ ]:
golden = load_golden_set_v3()
predictions = read_jsonl(V2_RUN_DIR / 'inference.jsonl')
old_details = read_jsonl(V2_RUN_DIR / 'scoring_details.jsonl')
old_summary = json.loads((V2_RUN_DIR / 'summary.json').read_text(encoding='utf-8'))

if len(golden) != len(predictions):
    raise ValueError(f'문항 수 불일치: golden={len(golden)}, predictions={len(predictions)}')
if {str(x['id']) for x in predictions} != set(golden['id'].astype(str)):
    raise ValueError('v2 예측 ID와 현재 골든셋 ID가 다릅니다.')

new_details, new_summary = scorer_v3.evaluate(
    golden.to_dict('records'), predictions
)
if new_summary.get('scorer_version') != EXPECTED_SCORER_VERSION:
    raise RuntimeError('메모리의 재채점 결과가 최신 채점기 버전이 아닙니다. Kernel을 재시작하고 Run All 하세요.')
print('재채점 완료:', len(new_details), '문항')

In [ ]:
if new_summary.get('scorer_version') != EXPECTED_SCORER_VERSION:
    raise RuntimeError('이전 셀의 오래된 결과가 남아 있습니다. Kernel을 재시작하고 Run All 하세요.')
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = PROJECT_ROOT / 'output' / 'scoring_v3_runs' / run_id
run_dir.mkdir(parents=True, exist_ok=False)

details_jsonl = run_dir / 'scoring_details.jsonl'
with details_jsonl.open('w', encoding='utf-8') as handle:
    for row in new_details:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
pd.DataFrame(new_details).to_csv(run_dir / 'scoring_details.csv', index=False, encoding='utf-8-sig')
(run_dir / 'summary.json').write_text(json.dumps(new_summary, ensure_ascii=False, indent=2), encoding='utf-8')

old_by_id = {str(row['id']): row for row in old_details}
comparison = []
for row in new_details:
    old = old_by_id[str(row['id'])]
    comparison.append({
        'id': row['id'],
        'v2_status': old.get('response_status'),
        'v3_status': row.get('response_status'),
        'v2_score': old.get('end_to_end_score'),
        'v3_score': row.get('end_to_end_score'),
        'score_delta': (row.get('end_to_end_score') - old.get('end_to_end_score') if row.get('end_to_end_score') is not None and old.get('end_to_end_score') is not None else None),
    })
comparison_df = pd.DataFrame(comparison)
changed_mask = (comparison_df.v2_status != comparison_df.v3_status) | (comparison_df.v2_score != comparison_df.v3_score)
comparison_df.to_csv(run_dir / 'v2_v3_comparison.csv', index=False, encoding='utf-8-sig')

summary_comparison = {
    'input_v2_run_id': V2_RUN_ID,
    'v2': old_summary,
    'v3': new_summary,
    'changed_case_count': int(changed_mask.sum()),
    'changed_cases': comparison_df.loc[changed_mask, 'id'].tolist(),
}
(run_dir / 'v2_v3_summary_comparison.json').write_text(json.dumps(summary_comparison, ensure_ascii=False, indent=2), encoding='utf-8')

def git_text(*args):
    return subprocess.check_output(['git', *args], cwd=PROJECT_ROOT, text=True).strip()

manifest = {
    'run_id': run_id,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_v2_run_id': V2_RUN_ID,
    'input_inference_sha256': sha256_file(V2_RUN_DIR / 'inference.jsonl'),
    'v3_base_sha256': sha256_file(V3_BASE_PATH),
    'v3_scorer_sha256': sha256_file(V3_SCORER_PATH),
    'v3_scorer_version': scorer_v3.SCORER_VERSION,
    'git_commit': git_text('rev-parse', 'HEAD'),
    'git_branch': git_text('branch', '--show-current'),
    'generation_api_called': False,
    'inference_reused_from_v2': True,
}
(run_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

display(comparison_df[changed_mask])
print(json.dumps(summary_comparison, ensure_ascii=False, indent=2))
print('결과 폴더:', run_dir)